# Plotting  

## Introduction

This tutorial describes **skrf's** plotting features. To plot with plotly instead of matplotlib, set `rf.plotting.options.backend = 'plotly'`.  If you would like to use skrf's matplotlib interface with skrf styling, start with this

In [ ]:
%matplotlib inline

In [ ]:
from matplotlib import pyplot as plt  # for advanced smith chart only

import skrf as rf

## Plotting Methods


Plotting functions are implemented as methods of the `Network` class.

* `Network.plot_s_re`
* `Network.plot_s_im`
* `Network.plot_s_mag`
* `Network.plot_s_db`
* ...
    
Similar methods exist for Impedance (`Network.z`) and Admittance Parameters (`Network.y`), 


* `Network.plot_z_re`
* `Network.plot_z_im`
* ...
* `Network.plot_y_re`
* `Network.plot_y_im`
* ...




## Smith Chart 


As a first example, load a [Network](../api/network.rst) and plot all four  s-parameters on the Smith chart.

In [ ]:
from skrf import Network

ring_slot = Network('data/ring slot.s2p')
ring_slot.plot_s_smith()

In [ ]:
# the same plot with plotly
ring_slot.plot_s_smith(backend='plotly')

scikit-rf includes a convenient command to make nicer figures quick:

In [ ]:
rf.stylely()  # nicer looking. Can be configured with different styles
ring_slot.plot_s_smith()

In [ ]:
# rf.stylely() only styles matplotlib, plotly figures use plotly's templates
ring_slot.plot_s_smith(backend='plotly')

In [ ]:
ring_slot.plot_s_smith(draw_labels=True)

In [ ]:
# the same plot with plotly
ring_slot.plot_s_smith(draw_labels=True, backend='plotly')

Another common option is to draw admittance contours, instead of impedance. This is controlled through the  `chart_type` argument.

In [ ]:
ring_slot.plot_s_smith(chart_type='y')

In [ ]:
# the same plot with plotly
ring_slot.plot_s_smith(chart_type='y', backend='plotly')

### Advanced Smith Chart with markers

The Smith Chart is a convenient plot that shows complex numbers with real and imaginary parts from zero to infinity with a zoom on a real reference in the center region. It is, however, not straightforward to figure out what point is related to which frequency because there is no frequency axis. A common workaround is to provide some frequency markers related to a table.

In [ ]:
# prepare markers
lines = [
    {'marker_idx': [30, 60, 90], 'color': 'g', 'm': 0, 'n': 0, 'ntw': ring_slot},
    {'marker_idx': [15, 45, 75], 'color': 'r', 'm': 1, 'n': 0, 'ntw': ring_slot},
]

# prepare figure
fig, ax = plt.subplots(1, 1, figsize=(7,8))

# impedance smith chart
rf.plotting.smith(ax = ax, draw_labels = True, ref_imm = 50.0, chart_type = 'z')

# plot data
col_labels = ['Frequency', 'Real Imag']
row_labels = []
row_colors = []
cell_text = []
for l in lines:
    m = l['m']
    n = l['n']
    l['ntw'].plot_s_smith(m=m, n=n, ax = ax, color=l['color'])
    #plot markers
    for i, k in enumerate(l['marker_idx']):
        x = l['ntw'].s.real[k, m, n]
        y = l['ntw'].s.imag[k, m, n]
        z = l['ntw'].z[k, m, n]
        z = f'{z.real:.4f} + {z.imag:.4f}j ohm'
        f = l['ntw'].frequency.f_scaled[k]
        f_unit = l['ntw'].frequency.unit
        row_labels.append(f'M{i + 1}')
        row_colors.append(l['color'])
        ax.scatter(x, y, marker = 'v', s=20, color=l['color'])
        ax.annotate(row_labels[-1], (x, y), xytext=(-7, 7), textcoords='offset points', color=l['color'])
        cell_text.append([f'{f:.3f} {f_unit}', z])
leg1 = ax.legend(loc="upper right", fontsize= 6)

# plot the table
the_table = ax.table(cellText=cell_text,
                      colWidths=[0.4] * 2,
                      rowLabels=row_labels,
                      colLabels=col_labels,
                      rowColours=row_colors,
                      loc='bottom')
the_table.auto_set_font_size(False)
the_table.set_fontsize(6)
#the_table.scale(1.5, 1.5)

In [ ]:
# the same chart with plotly
import plotly.graph_objects as go

fig = rf.plotting.smith(draw_labels=True, ref_imm=50.0, chart_type='z', backend='plotly')

# plot data
row_labels = []
row_colors = []
cell_text = []
for l in lines:
    m = l['m']
    n = l['n']
    s = l['ntw'].s[:, m, n]
    color = {'g': 'green', 'r': 'red'}[l['color']]
    fig.add_scatter(x=s.real, y=s.imag, mode='lines', line_color=color, name=f"{l['ntw'].name}, S{m + 1}{n + 1}")
    #plot markers
    labels = [f'M{i + 1}' for i in range(len(l['marker_idx']))]
    fig.add_scatter(x=s.real[l['marker_idx']], y=s.imag[l['marker_idx']], mode='markers+text', text=labels,
                    textposition='top left', textfont_color=color, showlegend=False,
                    marker=dict(symbol='triangle-down', size=8, color=color))
    for label, k in zip(labels, l['marker_idx']):
        z = l['ntw'].z[k, m, n]
        row_labels.append(label)
        row_colors.append(color)
        cell_text.append([f"{l['ntw'].frequency.f_scaled[k]:.3f} {l['ntw'].frequency.unit}",
                          f'{z.real:.4f} + {z.imag:.4f}j ohm'])

# plot the table below the chart
fig.add_trace(go.Table(header=dict(values=['', *col_labels]),
                       cells=dict(values=[row_labels, *zip(*cell_text)], fill_color=[row_colors, 'white', 'white'],
                                  font_color=['white', 'black', 'black']),
                       columnwidth=[1, 4, 4], domain=dict(y=[0, 0.3])))
fig.update_yaxes(domain=[0.35, 1])
fig.update_layout(height=800, legend=dict(x=1, y=1, xanchor='right'))

### Advanced Smith Chart with background

The Smith Chart can be drawn above a high-resolution Smith Chart background (or another image upon your fantasy).

In [ ]:
# prepare figure
fig, ax = plt.subplots(1, 1, figsize=(8,8))
background = plt.imread('figures/smithchart.png')

# tweak background position
ax.imshow(background, extent=[-1.185, 1.14, -1.13, 1.155])
rf.plotting.smith(ax = ax, draw_labels = True, ref_imm = 1.0, chart_type = 'z')

ring_slot.plot_s_smith(ax = ax)

In [ ]:
# the same chart with plotly
from PIL import Image

fig = ring_slot.plot_s_smith(draw_labels=True, backend='plotly')

# tweak background position
fig.add_layout_image(source=Image.open('figures/smithchart.png'), xref='x', yref='y', x=-1.185, y=1.155,
                     sizex=2.325, sizey=2.285, sizing='stretch', layer='below')
fig.update_xaxes(range=[-1.2, 1.2])
fig.update_yaxes(range=[-1.2, 1.2])
fig.update_layout(width=800, height=800)

    
See `skrf.plotting.smith()` for more info on customizing the Smith Chart. 

## Complex Plane 


Network parameters can also be plotted in the complex plane without a Smith Chart through `Network.plot_s_complex`.

In [ ]:
from matplotlib import pyplot as plt

ring_slot.plot_s_complex()

plt.axis('equal') # otherwise circles won't be circles

In [ ]:
# the same plot with plotly, which keeps circles round by itself
ring_slot.plot_s_complex(backend='plotly')

## Log-Magnitude 


Scalar components of the complex network parameters can be plotted vs 
frequency as well. To plot the log-magnitude of the s-parameters vs. frequency,

In [ ]:
ring_slot.plot_s_db()

In [ ]:
# the same plot with plotly
ring_slot.plot_s_db(backend='plotly')

    
When no arguments are passed to the plotting methods, all parameters are plotted. Single parameters can be plotted by passing indices `m` and `n` to the plotting commands (indexing start from 0). Comparing the simulated reflection coefficient off the ring slot to a measurement, 

In [ ]:
from skrf.data import ring_slot_meas

ring_slot.plot_s_db(m=0,n=0, label='Theory')
ring_slot_meas.plot_s_db(m=0,n=0, label='Measurement')

In [ ]:
# the same plot with plotly: each call returns its own figure, so add the measurement to the first one
fig = ring_slot.plot_s_db(m=0, n=0, label='Theory', backend='plotly')
fig.add_traces(ring_slot_meas.plot_s_db(m=0, n=0, label='Measurement', backend='plotly').data)

## Phase 


Plot phase, 

In [ ]:
ring_slot.plot_s_deg()

In [ ]:
# the same plot with plotly
ring_slot.plot_s_deg(backend='plotly')

    
Or unwrapped phase, 

In [ ]:
ring_slot.plot_s_deg_unwrap()

In [ ]:
# the same plot with plotly
ring_slot.plot_s_deg_unwrap(backend='plotly')

Phase is radian (rad) is also available

## Group Delay 

A Network has a `plot()` method which creates a rectangular plot of the argument vs frequency.  This can be used to make plots are aren't 'canned'. For example group delay

In [ ]:
gd = abs(ring_slot.s21.group_delay) *1e9 # in ns

ring_slot.plot(gd)
plt.ylabel('Group Delay (ns)')
plt.title('Group Delay of Ring Slot S21')

In [ ]:
# the same plot with plotly
fig = ring_slot.plot(gd, backend='plotly')
fig.update_layout(title='Group Delay of Ring Slot S21', yaxis_title='Group Delay (ns)')

## Impedance, Admittance 


The components the Impedance and Admittance parameters can be plotted 
similarly, 

In [ ]:
ring_slot.plot_z_im()

In [ ]:
# the same plot with plotly
ring_slot.plot_z_im(backend='plotly')

In [ ]:
ring_slot.plot_y_im()

In [ ]:
# the same plot with plotly
ring_slot.plot_y_im(backend='plotly')

## Customizing Plots


The legend entries are automatically filled in with the Network's `Network.name`. The entry can be overridden by passing the `label` argument to the plot method.

In [ ]:
ring_slot.plot_s_db(m=0,n=0, label = 'Simulation')

In [ ]:
# the same plot with plotly
ring_slot.plot_s_db(m=0, n=0, label='Simulation', backend='plotly')

The frequency unit used on the x-axis is automatically filled in from 
the Networks `Network.frequency.unit` attribute. To change
the label, change the frequency's `unit`.

In [ ]:
ring_slot.frequency.unit = 'mhz'
ring_slot.plot_s_db(0,0)

In [ ]:
# the same plot with plotly
ring_slot.plot_s_db(0, 0, backend='plotly')

Other key word arguments given to the plotting methods are passed through to the matplotlib `matplotlib.pyplot.plot` function. 

In [ ]:
ring_slot.frequency.unit='ghz'
ring_slot.plot_s_db(m=0,n=0, linewidth = 3, linestyle = '--', label = 'Simulation')
ring_slot_meas.plot_s_db(m=0,n=0, marker = 'o', markevery = 10,label = 'Measured')


In [ ]:
# the same plot with plotly, where marker.maxdisplayed thins the markers like markevery
fig = ring_slot.plot_s_db(m=0, n=0, linewidth=3, linestyle='--', label='Simulation', backend='plotly')
fig.add_traces(ring_slot_meas.plot_s_db(m=0, n=0, label='Measured', backend='plotly',
                                        marker=dict(symbol='circle', maxdisplayed=len(ring_slot_meas) // 10)).data)

All components of the plots can be customized through  [matplotlib]( http://matplotlib.sourceforge.net) functions, and `styles` can be used with a context manager. 

In [ ]:
from matplotlib import pyplot as plt
from matplotlib import style

mpl_style = "seaborn-ticks"
mpl_style = mpl_style if mpl_style in style.available else "seaborn-v0_8-ticks"

with style.context(mpl_style):
    ring_slot.plot_s_smith()
    plt.xlabel('Real Part')
    plt.ylabel('Imaginary Part')
    plt.title('Smith Chart With Legend Room')
    plt.axis([-1.1,2.1,-1.1,1.1])
    plt.legend(loc=5)


In [ ]:
# the same chart with plotly, using a plotly template in place of the matplotlib style
fig = ring_slot.plot_s_smith(backend='plotly')
fig.update_layout(template='simple_white', title=dict(text='Smith Chart With Legend Room', x=0.5),
                  xaxis_title='Real Part', yaxis_title='Imaginary Part', margin_t=60,
                  legend=dict(x=1, y=0.5, xanchor='right', yanchor='middle'))
fig.update_xaxes(range=[-1.1, 2.1], showline=False, ticks='')
fig.update_yaxes(range=[-1.1, 1.1], showline=False, ticks='')

## Saving Plots

Plots can be saved in various file formats using the GUI provided by the matplotlib. However, skrf provides a convenience function, called `skrf.plotting.save_all_figs`,  that allows all open figures to be saved to disk in multiple file formats, with filenames pulled from each figure's title,

In [ ]:
from skrf.plotting import save_all_figs

save_all_figs('data/', format=['png','eps','pdf'])

In [ ]:
# like save_all_figs here, this saves nothing: plotly keeps no list of open figures.
# Save a figure itself with fig.write_html('data/name.html'), or with fig.write_image('data/name.png')
# (png, pdf, ...), which needs the kaleido package

## Adding Markers Post Plot


A common need is to make a color plot, interpretable in greyscale print. 
The `skrf.plotting.add_markers_to_lines`  adds different markers each line in a plots *after* the plot has been made, which is usually when you remember to add them.

In [ ]:
from skrf import plotting

with plt.style.context('grayscale'):
    ring_slot.plot_s_deg()
    plotting.add_markers_to_lines()
    plt.legend() # have to re-generate legend


In [ ]:
# the same plot with plotly, setting the grayscale look on the figure
fig = ring_slot.plot_s_deg(backend='plotly')
for trace, color, symbol in zip(fig.data, ['black', '#666666', '#999999', '#b3b3b3'],
                                ['circle', 'diamond', 'square', 'cross']):
    trace.update(line_color=color, mode='lines+markers',
                 marker=dict(symbol=symbol, maxdisplayed=len(ring_slot) // 10))
fig.update_xaxes(gridcolor='black')
fig.update_yaxes(gridcolor='black')
fig.update_layout(plot_bgcolor='white', paper_bgcolor='#bfbfbf')

## Plotting Functions

The `Network` plotting methods are built on general functions in `skrf.plotting`, which plot plain arrays. Most of them take the same `backend` argument. `shade_bands` and `plot_vector` draw on an existing matplotlib plot, so with plotly the same is done with plotly's own methods on the returned figure.

### Rectangular, Polar, Complex Plane and Smith Chart

In [ ]:
from skrf import plotting

s11 = ring_slot.s[:, 0, 0]
plotting.plot_rectangular(ring_slot.frequency.f_scaled, abs(s11), x_label='Frequency (GHz)',
                          y_label='|S11|', title='Rectangular', label='S11')

In [ ]:
# the same plot with plotly
plotting.plot_rectangular(ring_slot.frequency.f_scaled, abs(s11), x_label='Frequency (GHz)',
                          y_label='|S11|', title='Rectangular', label='S11', backend='plotly')

In [ ]:
plotting.plot_complex_polar(s11, title='Polar', label='S11')

In [ ]:
# the same plot with plotly
plotting.plot_complex_polar(s11, title='Polar', label='S11', backend='plotly')

In [ ]:
plotting.plot_complex_rectangular(s11, label='S11')

In [ ]:
# the same plot with plotly
plotting.plot_complex_rectangular(s11, label='S11', backend='plotly')

In [ ]:
plotting.plot_smith(s11, draw_labels=True, label='S11')

In [ ]:
# the same plot with plotly
plotting.plot_smith(s11, draw_labels=True, label='S11', backend='plotly')

### Subplots

`skrf.plotting.subplot_params` plots each parameter of a network on its own subplot.

In [ ]:
fig, axs = plotting.subplot_params(ring_slot)

In [ ]:
# the same plot with plotly
plotting.subplot_params(ring_slot, backend='plotly')

### Shading Frequency Bands

`skrf.plotting.shade_bands` shades the frequency bands between the given edges, in Hz.

In [ ]:
ring_slot.plot_s_db(m=1, n=0)
plotting.shade_bands([75e9, 85e9, 95e9, 110e9], alpha=0.2)

In [ ]:
# the same plot with plotly, shading each band with fig.add_vrect in the figure's frequency unit (GHz)
fig = ring_slot.plot_s_db(m=1, n=0, backend='plotly')
for start, stop, color in [(75, 85, 'red'), (85, 95, 'blue'), (95, 110, 'yellow')]:
    fig.add_vrect(x0=start, x1=stop, fillcolor=color, opacity=0.2, line_width=0, layer='below')
fig

### Vectors

`skrf.plotting.plot_vector` draws a complex number as an arrow, here S11 at 92.5 GHz on a Smith chart.

In [ ]:
ring_slot.plot_s_smith(m=0, n=0)
plotting.plot_vector(s11[100], color='r')

In [ ]:
# the same plot with plotly, drawing the arrow from the origin with fig.add_annotation
fig = ring_slot.plot_s_smith(m=0, n=0, backend='plotly')
fig.add_annotation(x=s11[100].real, y=s11[100].imag, ax=0, ay=0, xref='x', yref='y', axref='x', ayref='y',
                   text='', showarrow=True, arrowhead=2, arrowwidth=2, arrowcolor='red')

### Contours

`skrf.plotting.plot_contour` draws contours of values given at scattered reflection coefficients, such as load-pull or noise measurements, and returns the point with the minimum or maximum value. Here, the gain of loads matched to a source with reflection coefficient `gamma_s` is computed on a grid of points:

In [ ]:
import numpy as np

r, theta = np.meshgrid(np.linspace(0.05, 0.95, 19), np.linspace(0, 2 * np.pi, 36, endpoint=False))
gamma = (r * np.exp(1j * theta)).ravel()
gamma_s = 0.3 + 0.2j
gain_db = 10 * np.log10(1 - abs((gamma - gamma_s.conjugate()) / (1 - gamma * gamma_s))**2)

gamma_opt, gain_max = plotting.plot_contour(1e9, gamma.real, gamma.imag, gain_db, min0max1=1,
                                            title='Mismatch Gain (dB)')
print(gamma_opt.s[0, 0, 0], gain_max)

In [ ]:
# the same plot with plotly
gamma_opt, gain_max = plotting.plot_contour(1e9, gamma.real, gamma.imag, gain_db, min0max1=1,
                                            title='Mismatch Gain (dB)', backend='plotly')
print(gamma_opt.s[0, 0, 0], gain_max)

## Calibration Plots

An overdetermined calibration, with more standards or more connections of each standard than it needs, shows how well its measurements agree. `Calibration.plot_residuals` plots the residuals of the corrected standards, and `Calibration.plot_calibration_errors` plots their biased, unbiased and total error (see `Calibration.biased_error`).

Here, a one-port calibration uses four standards, each connected three times with some noise on every connection, and a load whose actual reflection differs from its ideal:

In [ ]:
from skrf.calibration import OnePort
from skrf.media import DefinedGammaZ0

media = DefinedGammaZ0(ring_slot.frequency)
rng = np.random.default_rng(0)

def connect(ntwk, name):
    """One connection of a standard: `ntwk` with some noise, named after its ideal."""
    ntwk = ntwk.copy()
    ntwk.s += 0.005 * (rng.standard_normal(ntwk.s.shape) + 1j * rng.standard_normal(ntwk.s.shape))
    ntwk.name = name
    return ntwk

error = media.random(n_ports=2, name='error')
ideals = [media.short(name='short'), media.open(name='open'), media.load(0.1, name='load'),
          media.delay_short(45, 'deg', name='delay short')]
actuals = [ideals[0], ideals[1], media.load(0.15), ideals[3]]

measured = [connect(error ** actual, ideal.name) for _ in range(3) for actual, ideal in zip(actuals, ideals)]
cal = OnePort(ideals=ideals * 3, measured=measured)
cal.run()

### Residuals

`Calibration.plot_residuals` plots the residual of every connection:

In [ ]:
cal.plot_residuals()

In [ ]:
# the same plot with plotly
cal.plot_residuals(backend='plotly')

Its `attr` argument selects the plotted property, here the residuals on a Smith chart:

In [ ]:
cal.plot_residuals('s_smith')

In [ ]:
# the same plot with plotly
cal.plot_residuals('s_smith', backend='plotly')

### Calibration Errors

`Calibration.plot_calibration_errors` plots the biased, unbiased and total error in dB:

In [ ]:
cal.plot_calibration_errors()

In [ ]:
# the same plot with plotly
cal.plot_calibration_errors(backend='plotly')

For a two-port calibration, the errors of all port pairs share one plot, and the legend names the port pair of each trace:

In [ ]:
from skrf.calibration import TwelveTerm
from skrf.network import two_port_reflect

port1, port2 = media.random(n_ports=2), media.random(n_ports=2)
ideals = [two_port_reflect(media.short(), media.short()), two_port_reflect(media.open(), media.open()),
          two_port_reflect(media.load(0.1), media.load(0.1)), media.thru()]
for ideal, name in zip(ideals, ['short', 'open', 'load', 'thru']):
    ideal.name = name

measured = [connect(port1 ** ideal ** port2, ideal.name) for _ in range(2) for ideal in ideals]
cal2 = TwelveTerm(ideals=ideals * 2, measured=measured, n_thrus=1)
cal2.run()
cal2.plot_calibration_errors()

In [ ]:
# the same plot with plotly
cal2.plot_calibration_errors(backend='plotly')

## Material Properties

`Freespace.plot_ep`, `plot_mu` and `plot_ep_mu` plot the real and imaginary parts of the relative permittivity and permeability of a `Freespace` medium, here a lossy dielectric whose loss grows with frequency:

In [ ]:
from skrf.media import Freespace

f = ring_slot.frequency
dielectric = Freespace(f, ep_r=4.2 - 0.08j * f.f / f.f[0], mu_r=1.1)
dielectric.plot_ep_mu()

In [ ]:
# the same plot with plotly
dielectric.plot_ep_mu(backend='plotly')